# 01 — Exploración de datasets INEGI (Camino A)

Carga, inspección y generación de los CSV procesados.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

## 1. ATUS — inspección rápida

In [ ]:
from src.data_processing import listar_archivos_atus
archivos = listar_archivos_atus()
print(f'Archivos anuales encontrados: {len(archivos)}')
print('Rango:', archivos[0].name, '→', archivos[-1].name)

In [ ]:
# Muestra del archivo más reciente.
df_2024 = pd.read_csv(archivos[-1], nrows=3, encoding='utf-8')
df_2024.head()

In [ ]:
# Detección del bug ANIO=1 en archivos antiguos.
import re
for p in archivos:
    yr_filename = int(re.search(r'_(\d{4})\.csv$', p.name).group(1))
    sample = pd.read_csv(p, nrows=3, encoding='utf-8', usecols=['ANIO'])
    coincide = (sample['ANIO'] == yr_filename).all()
    print(f'  {yr_filename}: ANIO en CSV = {sample["ANIO"].iloc[0]:>5}  {"OK" if coincide else "<-- CORRUPTO"}')

## 2. VMRC — inspección rápida

In [ ]:
from src.data_processing import (
    VMRC_VENTA_AUTO, VMRC_VENTA_CAMION, VMRC_VEHICULOS, VMRC_PRODUCCION,
)
for p in (VMRC_VENTA_AUTO, VMRC_VENTA_CAMION, VMRC_VEHICULOS, VMRC_PRODUCCION):
    df = pd.read_csv(p, nrows=2, encoding='utf-8')
    print(f'\n=== {p.name}')
    print(f'  Shape (sample): {df.shape}, COBERTURA: {df["COBERTURA"].iloc[0] if "COBERTURA" in df.columns else "N/A"}')
    print(f'  Cols: {list(df.columns)}')

## 3. Generar todos los procesados

In [ ]:
from src.data_processing import construir_procesados
construir_procesados()

## 4. Verificar procesados

In [ ]:
from src.data_processing import (
    load_atus_anual, load_atus_detalle, load_vmrc_ventas, load_vmrc_parque,
)
atus_anual = load_atus_anual()
atus_detalle = load_atus_detalle()
vmrc_ventas = load_vmrc_ventas()
vmrc_parque = load_vmrc_parque()

print('atus_anual   :', atus_anual.shape, '— años', atus_anual['anio'].min(), '-', atus_anual['anio'].max())
print('atus_detalle :', atus_detalle.shape)
print('vmrc_ventas  :', vmrc_ventas.shape, '— tipos', sorted(vmrc_ventas['tipo'].unique()))
print('vmrc_parque  :', vmrc_parque.shape)

In [ ]:
# Sanity check: número de entidades y años en ATUS
print('Entidades en ATUS:', atus_anual['entidad'].nunique())
print('Años en ATUS:', sorted(atus_anual['anio'].unique()))

## 5. Probar funciones de analytics

In [ ]:
from src import analytics

print('Top 5 estados con más accidentes en 2023:')
display(analytics.ranking_accidentes(atus_anual, 2023, 5))

In [ ]:
print('Tendencia de ventas nacionales (auto):')
display(analytics.tendencia_ventas_nacional(vmrc_ventas, 'auto').tail(10))

In [ ]:
print('Relación nacional ventas vs accidentes 2015-2023:')
display(analytics.relacion_ventas_accidentes_nacional(atus_anual, vmrc_ventas, 2015, 2023, 'total'))